In [1]:
# 1. Importando bibliotecas e carregando os dados brutos
import pandas as pd
import numpy as np

# Carregando o dataset original (intacto)
caminho_raw = '../data/raw/food_delivery_orders_dataset.csv'
df_bruto = pd.read_csv(caminho_raw)

print(f"Tamanho original: {df_bruto.shape[0]} linhas e {df_bruto.shape[1]} colunas.")

Tamanho original: 50000 linhas e 37 colunas.


In [2]:
# 2. Tratamento de Valores Ausentes e Criação do Dataset Limpo

# Criamos uma cópia para não alterar o df_bruto na memória
df_limpo = df_bruto.copy()

# 2.1 Removendo a coluna 'cancellation_reason' (98% nulos)
df_limpo = df_limpo.drop(columns=['cancellation_reason'])

# 2.2 Removendo as linhas (pedidos) cancelados (que geraram os outros nulos)
df_limpo = df_limpo.dropna()

print(f"Tamanho após limpeza: {df_limpo.shape[0]} linhas e {df_limpo.shape[1]} colunas.")

# 2.3 Salvando o novo dataset na pasta 'processed'
caminho_processed = '../data/processed/dataset_clean.csv'
df_limpo.to_csv(caminho_processed, index=False)
print(f"Dataset limpo salvo com sucesso em: {caminho_processed}")

Tamanho após limpeza: 49132 linhas e 36 colunas.
Dataset limpo salvo com sucesso em: ../data/processed/dataset_clean.csv


In [3]:
# 3. Tratamento de Outliers (Capping / Limitação)
# Definimos um limite máximo realista para o tempo de entrega (ex: 120 minutos ou 2 horas)
limite_maximo = 120

# A função np.where funciona como um "SE" do Excel:
# SE o tempo for maior que 120, vira 120. SENÃO, mantém o valor original.
df_limpo['actual_delivery_time_minutes'] = np.where(
    df_limpo['actual_delivery_time_minutes'] > limite_maximo, 
    limite_maximo, 
    df_limpo['actual_delivery_time_minutes']
)

print("Outliers de tempo de entrega limitados a 120 minutos com sucesso!")

Outliers de tempo de entrega limitados a 120 minutos com sucesso!


In [4]:
# 4. Verificação de valores negativos inválidos
# Pegamos apenas as colunas que têm números
colunas_numericas = df_limpo.select_dtypes(include=['int64', 'float64'])

# Criamos uma checagem de quantos valores são menores que zero
valores_negativos = (colunas_numericas < 0).sum()

# Mostramos apenas as colunas que têm algum valor negativo
print("Colunas com valores negativos:")
print(valores_negativos[valores_negativos > 0])

Colunas com valores negativos:
Series([], dtype: int64)


In [5]:
# 5. Tratamento de valores negativos (Anomalias)
# Vamos usar a coluna 'distance_km' como exemplo.
# Calculamos a mediana apenas considerando os valores válidos (maiores ou iguais a zero)
mediana_distancia = df_limpo.loc[df_limpo['distance_km'] >= 0, 'distance_km'].median()

# Substituímos os valores menores que zero pela mediana calculada
df_limpo['distance_km'] = np.where(
    df_limpo['distance_km'] < 0, 
    mediana_distancia, 
    df_limpo['distance_km']
)

print(f"Possíveis valores negativos corrigidos para a mediana: {mediana_distancia:.2f} km")

Possíveis valores negativos corrigidos para a mediana: 4.10 km


In [6]:
# 6. Codificação de Variáveis Categóricas (One-Hot Encoding)
# Escolhendo colunas de texto que são importantes para as entregas
colunas_para_codificar = ['weather', 'traffic_level', 'customer_type', 'day_of_week']

# O get_dummies transforma os textos em colunas de 0 (Falso) e 1 (Verdadeiro)
# drop_first=True remove uma das opções para evitar informações repetidas (ex: se não é dia, é noite)
df_limpo = pd.get_dummies(df_limpo, columns=colunas_para_codificar, drop_first=True)

# Convertendo os booleanos (True/False) que o pandas gera para números inteiros (1 e 0)
for col in df_limpo.columns:
    if df_limpo[col].dtype == bool:
        df_limpo[col] = df_limpo[col].astype(int)

print(f"Novo tamanho do dataset: {df_limpo.shape[0]} linhas e {df_limpo.shape[1]} colunas.")

Novo tamanho do dataset: 49132 linhas e 45 colunas.


In [7]:
# 7. Normalização (Min-Max Scaling)
from sklearn.preprocessing import MinMaxScaler

# 7.1 Selecionamos apenas os Nomes das colunas que são puramente numéricas
colunas_numericas = df_limpo.select_dtypes(include=['int64', 'float64']).columns

# 7.2 Agora sim, filtramos apenas as que têm valor máximo maior que 1
colunas_para_escalar = [col for col in colunas_numericas if df_limpo[col].max() > 1]

# Inicializamos a ferramenta matemática
scaler = MinMaxScaler()

# Aplicamos a transformação
df_limpo[colunas_para_escalar] = scaler.fit_transform(df_limpo[colunas_para_escalar])

# Salvamos a versão final processada
caminho_final = '../data/processed/dataset_preprocessed.csv'
df_limpo.to_csv(caminho_final, index=False)

print(f"Normalização concluída! Arquivo salvo em: {caminho_final}")
display(df_limpo.head(3))

Normalização concluída! Arquivo salvo em: ../data/processed/dataset_preprocessed.csv


,order_id,customer_id,restaurant_id,driver_id,order_timestamp,order_date,order_hour,is_weekend,city,delivery_area,...,traffic_level_Low,traffic_level_Medium,customer_type_Premium,customer_type_Regular,day_of_week_Monday,day_of_week_Saturday,day_of_week_Sunday,day_of_week_Thursday,day_of_week_Tuesday,day_of_week_Wednesday
0,ORD_000001,CUS_00001,RES_0008,DRV_0075,2025-10-05 15:54:00,2025-10-05,0.652174,1,City_C,City_C_Area_3,...,0,1,1,0,0,0,1,0,0,0
1,ORD_000002,CUS_00013,RES_0024,DRV_0131,2025-10-01 20:00:00,2025-10-01,0.869565,0,City_A,City_A_Area_1,...,0,0,0,1,0,0,0,0,0,1
2,ORD_000003,CUS_00049,RES_0013,DRV_0087,2025-10-07 18:24:00,2025-10-07,0.782609,0,City_C,City_C_Area_3,...,0,0,0,1,0,0,0,0,1,0
